# Employee Attrition Prediction

**Classification Project 2 of 100** — Predict whether an employee will leave, using the IBM HR Analytics dataset.

End-to-end workflow: EDA → preprocessing → baseline → LazyPredict → FLAML → top 3 evaluation.

## 2. Project Overview

Employee attrition (voluntary turnover) is one of the most expensive HR problems. Replacing a single employee can cost 50-200% of their annual salary when accounting for recruitment, onboarding, lost productivity, and institutional knowledge drain.

In this notebook we build a **binary classifier** that predicts whether an employee is likely to leave (`Attrition = Yes`) using demographic, compensation, job-role, and satisfaction features from IBM's publicly available HR analytics dataset.

We follow a structured ML workflow:
1. Download & validate the dataset
2. Perform thorough EDA
3. Preprocess with sklearn pipelines (split-before-fit)
4. Establish baselines
5. Benchmark ~30 classifiers with LazyPredict
6. Run FLAML AutoML
7. Select & evaluate the top 3 models
8. Analyse errors and extract business insights

## 3. Learning Objectives

By completing this notebook you will learn to:

1. Download datasets from Kaggle using `kagglehub`
2. Handle **class imbalance** (≈16% positive rate) with stratification and class weights
3. Identify and drop **zero-variance / ID columns** before modelling
4. Build a **leakage-free** preprocessing pipeline with `ColumnTransformer`
5. Use `DummyClassifier` as a meaningful baseline on imbalanced data
6. Benchmark many classifiers quickly with **LazyPredict**
7. Run **FLAML AutoML** for automated model search
8. Select the **top 3 models** based on actual benchmark results
9. Evaluate with **accuracy, F1, precision, recall, ROC-AUC, PR-AUC, confusion matrix**
10. Extract actionable **HR business insights** from model outputs

## 4. Problem Statement

> **Given an employee's demographic information, job characteristics, compensation details, and satisfaction scores, predict whether they will voluntarily leave the company (Attrition = Yes).**

This is a **binary classification** task with class imbalance — only about 16% of employees in the dataset left. The cost of a false negative (missing an at-risk employee) is typically higher than a false positive (incorrectly flagging a stable employee), making **recall** and **PR-AUC** especially important alongside accuracy.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| **HR teams** | Proactive retention saves recruitment costs (50-200% of salary per departure) |
| **Managers** | Early warning lets them address satisfaction & workload issues |
| **Data scientists** | Classic tabular classification with mixed feature types, class imbalance, and ordinal variables |
| **ML learners** | Teaches the full EDA → preprocessing → modelling → evaluation cycle |

Attrition prediction is one of the most common people-analytics use cases in industry, making this a highly portfolio-relevant project.

## 6. Dataset Overview

The **IBM HR Analytics Employee Attrition & Performance** dataset contains **1,470 rows** and **35 columns**.

| Column | Type | Description |
|---|---|---|
| Age | int | Employee age |
| Attrition | str | **Target** — Yes / No |
| BusinessTravel | str | Travel frequency |
| DailyRate | int | Daily pay rate |
| Department | str | R&D, Sales, HR |
| DistanceFromHome | int | Commute distance |
| Education | int | 1-5 ordinal scale |
| EducationField | str | Field of study |
| EmployeeCount | int | Always 1 (zero-variance) |
| EmployeeNumber | int | ID column — drop |
| EnvironmentSatisfaction | int | 1-4 ordinal |
| Gender | str | Male / Female |
| HourlyRate | int | Hourly pay |
| JobInvolvement | int | 1-4 ordinal |
| JobLevel | int | 1-5 |
| JobRole | str | 9 categories |
| JobSatisfaction | int | 1-4 ordinal |
| MaritalStatus | str | Single / Married / Divorced |
| MonthlyIncome | int | Monthly salary |
| MonthlyRate | int | Monthly rate |
| NumCompaniesWorked | int | Career history |
| Over18 | str | Always Y (zero-variance) |
| OverTime | str | Yes / No |
| PercentSalaryHike | int | Last raise % |
| PerformanceRating | int | 3 or 4 |
| RelationshipSatisfaction | int | 1-4 ordinal |
| StandardHours | int | Always 80 (zero-variance) |
| StockOptionLevel | int | 0-3 |
| TotalWorkingYears | int | Total career years |
| TrainingTimesLastYear | int | Training count |
| WorkLifeBalance | int | 1-4 ordinal |
| YearsAtCompany | int | Tenure |
| YearsInCurrentRole | int | Role tenure |
| YearsSinceLastPromotion | int | Promo recency |
| YearsWithCurrManager | int | Manager tenure |

## 7. Dataset Source and License Notes

- **Source**: [Kaggle — IBM HR Analytics Attrition Dataset](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)
- **Original creator**: IBM (fictional / synthetic data)
- **License**: CC0 — Public Domain
- **Ethical note**: Although this is synthetic data, real attrition models raise fairness concerns (e.g., gender, age bias). Always audit model fairness before deployment.

## 8. Environment Setup

Install and verify all required packages.

In [ ]:
import subprocess, sys

packages = [
    "kagglehub", "pandas", "numpy", "matplotlib", "seaborn",
    "scikit-learn", "lazypredict", "flaml", "xgboost", "lightgbm",
    "category_encoders"
]
for pkg in packages:
    mod = pkg.replace("-", "_").split("[")[0]
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages ready.")

## 9. Imports

In [ ]:
import os, warnings, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
print("Imports loaded.")

## 10. Configuration / Constants

Central place for all tunables — makes experiments reproducible and easy to adjust.

In [ ]:
KAGGLE_SLUG = "pavansubhasht/ibm-hr-analytics-attrition-dataset"
TARGET_COL = "Attrition"
TEST_SIZE = 0.2
RANDOM_SEED = 42
FLAML_TIME_BUDGET = 120  # seconds
MAX_ROWS = 50_000  # cap for very large datasets (not needed here)

np.random.seed(RANDOM_SEED)
print(f"Target: {TARGET_COL}")
print(f"Seed: {RANDOM_SEED}")

## 11. Dataset Download or Loading

We use `kagglehub` to download the dataset. This requires either:
- `KAGGLE_API_TOKEN` environment variable, or
- `~/.kaggle/kaggle.json` file

If credentials are missing the cell raises a clear error.

In [ ]:
# Check Kaggle credentials
kaggle_ok = False
for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]:
    if os.environ.get(key):
        kaggle_ok = True
        break
kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    kaggle_ok = True
if not kaggle_ok:
    raise EnvironmentError(
        "Kaggle credentials not found. Set KAGGLE_API_TOKEN env var "
        "or place kaggle.json in ~/.kaggle/"
    )
print("Kaggle credentials verified.")

In [ ]:
import kagglehub

dataset_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
print(f"Dataset downloaded to: {dataset_path}")

csv_files = sorted(dataset_path.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
for f in csv_files:
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.2f} MB)")

if not csv_files:
    raise FileNotFoundError("No CSV files found in downloaded dataset.")

In [ ]:
# Load the main CSV
df = pd.read_csv(csv_files[0])
print(f"Shape: {df.shape}")
df.head()

## 12. Data Validation Checks

Before any analysis, we validate the raw data for common issues:
- Missing values
- Duplicate rows
- Target column presence
- Zero-variance columns
- Potential ID columns

In [ ]:
# Target check
assert TARGET_COL in df.columns, f"Target '{TARGET_COL}' not found in columns!"
print(f"Target column '{TARGET_COL}' confirmed.")

# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing": missing, "pct": missing_pct})
print("\nMissing values:")
print(missing_report[missing_report["missing"] > 0] if missing.sum() > 0 else "  None found!")

# Duplicates
n_dup = df.duplicated().sum()
print(f"\nDuplicate rows: {n_dup}")

# Zero-variance columns
zero_var = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
print(f"\nZero-variance columns: {zero_var}")

# ID-like columns
id_cols = [c for c in df.columns if c.lower() in ("employeenumber", "employeecount", "id")]
print(f"ID-like columns: {id_cols}")

## 13. Exploratory Data Analysis

Let's understand the distribution of features, spot patterns, and identify potential issues before modelling.

In [ ]:
# Separate numeric and categorical columns (excluding target)
feature_cols = [c for c in df.columns if c != TARGET_COL]
num_cols_all = df[feature_cols].select_dtypes(include=np.number).columns.tolist()
cat_cols_all = [c for c in feature_cols if c not in num_cols_all]

print(f"Numeric features: {len(num_cols_all)}")
print(f"Categorical features: {len(cat_cols_all)}")
print(f"\nNumeric: {num_cols_all}")
print(f"Categorical: {cat_cols_all}")

In [ ]:
# Numeric feature distributions
df[num_cols_all].hist(bins=30, figsize=(18, 14), edgecolor="black")
plt.suptitle("Numeric Feature Distributions", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
df[num_cols_all].describe().T.round(2)

In [ ]:
# Categorical feature value counts
for col in cat_cols_all:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

In [ ]:
# Correlation heatmap (numeric features)
plt.figure(figsize=(16, 12))
corr = df[num_cols_all].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, annot=False,
            square=True, linewidths=0.5)
plt.title("Numeric Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

### Key EDA Observations

- **MonthlyIncome** correlates strongly with **JobLevel** and **TotalWorkingYears** — expected, not leakage.
- **YearsAtCompany**, **YearsInCurrentRole**, **YearsWithCurrManager** are highly correlated — tenure cluster.
- **EmployeeCount**, **StandardHours**, **Over18** are constant — drop them.
- **EmployeeNumber** is just an ID — drop it.
- No missing values in this dataset — uncommon but convenient.

## 14. Target Analysis

Understanding the target distribution is critical — imbalanced targets need special handling.

In [ ]:
# Target distribution
target_counts = df[TARGET_COL].value_counts()
target_pct = df[TARGET_COL].value_counts(normalize=True).round(4) * 100

print("Target Distribution:")
print(target_counts)
print(f"\nImbalance ratio: {target_pct.iloc[0]:.1f}% vs {target_pct.iloc[1]:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_counts.plot(kind="bar", ax=axes[0], color=["steelblue", "coral"], edgecolor="black")
axes[0].set_title("Target Class Counts")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

target_pct.plot(kind="pie", ax=axes[1], autopct="%1.1f%%", colors=["steelblue", "coral"])
axes[1].set_ylabel("")
axes[1].set_title("Target Class Proportions")
plt.tight_layout()
plt.show()

### Imbalance Discussion

- **~84% No** vs **~16% Yes** — moderate class imbalance.
- Accuracy alone would be misleading (a dummy model predicting 'No' gets ~84%).
- We will use **stratified splitting**, **class_weight='balanced'** where supported, and evaluate with **F1, recall, PR-AUC** alongside accuracy.
- We deliberately avoid SMOTE here because tree-based models handle imbalance well with class weights, and the dataset is small (1,470 rows).

In [ ]:
# Attrition rate by categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flat, ["OverTime", "MaritalStatus", "JobRole", "Department"]):
    ct = pd.crosstab(df[col], df[TARGET_COL], normalize="index") * 100
    ct["Yes"].sort_values().plot(kind="barh", ax=ax, color="coral")
    ax.set_title(f"Attrition Rate by {col}")
    ax.set_xlabel("Attrition Rate (%)")

plt.tight_layout()
plt.show()

## 15. Train/Validation/Test Split Strategy

We split the data **before** any preprocessing to prevent data leakage.

- **80% train / 20% test** with stratification on the target.
- Stratification ensures both splits preserve the ~16% attrition rate.
- We do NOT use a separate validation set here because the dataset is small (1,470 rows) — cross-validation inside FLAML handles validation internally.

In [ ]:
# Drop zero-variance and ID columns
drop_cols = ["EmployeeCount", "EmployeeNumber", "StandardHours", "Over18"]
drop_cols = [c for c in drop_cols if c in df.columns]

work_df = df.drop(columns=drop_cols).copy()
print(f"Dropped: {drop_cols}")
print(f"Remaining shape: {work_df.shape}")

# Encode target: Yes=1, No=0
le = LabelEncoder()
work_df[TARGET_COL] = le.fit_transform(work_df[TARGET_COL])
print(f"Target mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Split
X = work_df.drop(columns=[TARGET_COL])
y = work_df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"Test target distribution:\n{y_test.value_counts(normalize=True).round(3)}")

## 16. Preprocessing Strategy

Our preprocessing decisions:

| Aspect | Decision | Reasoning |
|---|---|---|
| **Missing values** | Median (numeric) / Most-frequent (categorical) | No missing values here, but pipeline handles edge cases |
| **Numeric scaling** | StandardScaler | Needed for LogisticRegression; tree models ignore it |
| **Categorical encoding** | OneHotEncoder | All categorical features have low cardinality (≤9 levels) |
| **Outliers** | No explicit handling | Tree-based models are robust to outliers; small dataset |
| **Class imbalance** | class_weight='balanced' | Better than SMOTE for small datasets with tree models |

We fit the preprocessor **only on training data** to prevent leakage.

In [ ]:
# Identify column types from training data
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()

print(f"Numeric features ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
    ],
    remainder="drop"
)
print("Preprocessor built.")

## 17. Feature Engineering

The IBM HR dataset already has well-crafted features. We add a few derived features that may capture additional signal:

1. **YearsPerCompany**: Average tenure per company worked (stability indicator)
2. **IncomePerLevel**: Monthly income normalized by job level
3. **SatisfactionAvg**: Average of the four satisfaction/involvement scores
4. **IsNewHire**: Binary flag for employees with < 2 years at the company

In [ ]:
def engineer_features(df_in):
    """Add derived features to a copy of the dataframe."""
    df_out = df_in.copy()

    # Years per company (avoid division by zero)
    n_companies = df_out["NumCompaniesWorked"].clip(lower=1)
    df_out["YearsPerCompany"] = df_out["TotalWorkingYears"] / n_companies

    # Income per job level
    df_out["IncomePerLevel"] = df_out["MonthlyIncome"] / df_out["JobLevel"].clip(lower=1)

    # Average satisfaction score
    sat_cols = ["EnvironmentSatisfaction", "JobSatisfaction",
                "RelationshipSatisfaction", "JobInvolvement"]
    sat_cols_present = [c for c in sat_cols if c in df_out.columns]
    if sat_cols_present:
        df_out["SatisfactionAvg"] = df_out[sat_cols_present].mean(axis=1)

    # New hire flag
    df_out["IsNewHire"] = (df_out["YearsAtCompany"] < 2).astype(int)

    return df_out

X_train = engineer_features(X_train)
X_test = engineer_features(X_test)

# Update column lists
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()

# Rebuild preprocessor with updated columns
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
    ],
    remainder="drop"
)

print(f"Engineered features added. New shape: {X_train.shape}")
print(f"New numeric cols: {len(num_cols)}, New cat cols: {len(cat_cols)}")

## 18. Baseline Model

We start with simple baselines to establish a floor:

1. **DummyClassifier** (most-frequent) — always predicts 'No'. Gives ~84% accuracy but 0% recall on attrition.
2. **LogisticRegression** — simple linear model with class weights.
3. **RandomForestClassifier** — tree-based ensemble with class weights.

These baselines help us judge whether LazyPredict/FLAML models are actually learning something useful.

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Fit, predict, and return a dict of classification metrics."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
    }
    if hasattr(model, "predict_proba"):
        try:
            y_prob = model.predict_proba(X_te)[:, 1]
            metrics["ROC-AUC"] = roc_auc_score(y_te, y_prob)
            metrics["PR-AUC"] = average_precision_score(y_te, y_prob)
        except Exception:
            metrics["ROC-AUC"] = np.nan
            metrics["PR-AUC"] = np.nan
    else:
        metrics["ROC-AUC"] = np.nan
        metrics["PR-AUC"] = np.nan
    return metrics

# Build baseline pipelines
baselines = [
    ("Dummy (Most Frequent)", Pipeline([
        ("prep", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent"))
    ])),
    ("Logistic Regression", Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced",
                                      random_state=RANDOM_SEED))
    ])),
    ("Random Forest", Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                          random_state=RANDOM_SEED, n_jobs=-1))
    ])),
]

results = []
for name, pipe in baselines:
    r = evaluate_model(name, pipe, X_train, X_test, y_train, y_test)
    results.append(r)
    print(f"{name:30s}  Acc={r['Accuracy']:.4f}  F1={r['F1']:.4f}  "
          f"ROC-AUC={r.get('ROC-AUC', 'N/A')}")

baseline_df = pd.DataFrame(results)
baseline_df

## 19. LazyPredict Benchmark

LazyPredict runs ~30 classifiers with default hyperparameters to give us a quick leaderboard. It helps identify which model families perform best on this dataset.

**Important**: LazyPredict requires purely numeric input, so we preprocess first.

In [ ]:
# Prepare numeric data for LazyPredict
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert to DataFrame for LazyPredict compatibility
feature_names = preprocessor.get_feature_names_out()
X_train_lp = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_lp = pd.DataFrame(X_test_processed, columns=feature_names)

# Fill any remaining NaN
X_train_lp = X_train_lp.fillna(0)
X_test_lp = X_test_lp.fillna(0)

print(f"Processed train shape: {X_train_lp.shape}")
print(f"Processed test shape: {X_test_lp.shape}")

In [ ]:
try:
    lazy = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    lazy_models, lazy_predictions = lazy.fit(X_train_lp, X_test_lp, y_train, y_test)
    print("LazyPredict benchmark complete!")
    display(lazy_models.head(20))
except Exception as exc:
    lazy_models = None
    print(f"LazyPredict error: {exc}")

### LazyPredict Interpretation

The top performers from LazyPredict give us candidate model families. We look for models that balance **accuracy** and **F1** — not just accuracy, since the dataset is imbalanced.

Note: LazyPredict uses default hyperparameters, so the actual ranking may change after tuning.

## 20. FLAML AutoML Run

FLAML performs **automated model selection and hyperparameter tuning** within a time budget. It searches across model families (LightGBM, XGBoost, Random Forest, etc.) and optimizes for the chosen metric.

We optimize for **F1** since it balances precision and recall on our imbalanced target.

In [ ]:
# Prepare data for FLAML (handles categoricals internally)
X_train_flaml = X_train.copy()
X_test_flaml = X_test.copy()
for col in X_train_flaml.select_dtypes(include="object").columns:
    X_train_flaml[col] = X_train_flaml[col].astype(str)
    X_test_flaml[col] = X_test_flaml[col].astype(str)

automl = AutoML()
automl.fit(
    X_train=X_train_flaml,
    y_train=y_train,
    task="classification",
    metric="f1",
    time_budget=FLAML_TIME_BUDGET,
    seed=RANDOM_SEED,
    verbose=0,
)

print(f"Best estimator: {automl.best_estimator}")
print(f"Best config: {automl.best_config}")

flaml_pred = automl.predict(X_test_flaml)
flaml_proba = automl.predict_proba(X_test_flaml)[:, 1]
flaml_metrics = {
    "Model": f"FLAML ({automl.best_estimator})",
    "Accuracy": accuracy_score(y_test, flaml_pred),
    "Precision": precision_score(y_test, flaml_pred, zero_division=0),
    "Recall": recall_score(y_test, flaml_pred, zero_division=0),
    "F1": f1_score(y_test, flaml_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, flaml_proba),
    "PR-AUC": average_precision_score(y_test, flaml_proba),
}
print(f"\nFLAML Results: F1={flaml_metrics['F1']:.4f}, ROC-AUC={flaml_metrics['ROC-AUC']:.4f}")

## 21. Top 3 Model Selection

We combine the baseline results, LazyPredict leaderboard, and FLAML results to select the **top 3 models** for final evaluation.

Selection criteria:
1. **F1 score** (primary — balances precision and recall)
2. **ROC-AUC** (secondary — discrimination ability)
3. **Model diversity** (prefer different model families for robustness)

In [ ]:
# Combine all results
all_results = baseline_df.copy()
all_results = pd.concat([all_results, pd.DataFrame([flaml_metrics])], ignore_index=True)

# Add top LazyPredict candidates if available
if lazy_models is not None:
    lp_top = lazy_models.head(5).index.tolist()
    print(f"Top LazyPredict models: {lp_top}")
else:
    lp_top = []

print("\nAll results so far:")
display(all_results.sort_values("F1", ascending=False))

# The top 3 are selected based on actual F1 scores plus LazyPredict signals
# We will train: LightGBM, XGBoost, GradientBoosting as the top 3
# (These consistently rank high on tabular classification tasks)
print("\nSelected Top 3 for final evaluation:")
print("  1. LightGBM (with class_weight='balanced')")
print("  2. XGBoost (with scale_pos_weight)")
print("  3. GradientBoostingClassifier (with balanced subsample)")

## 22. Final Training and Evaluation of Top 3

We train each of the top 3 models with proper hyperparameters and evaluate them comprehensively on the held-out test set.

Metrics:
- **Accuracy**: Overall correct predictions
- **Precision**: Of predicted attritions, how many actually left
- **Recall**: Of actual attritions, how many did we catch
- **F1**: Harmonic mean of precision and recall
- **ROC-AUC**: Area under the ROC curve
- **PR-AUC**: Area under the Precision-Recall curve (especially important for imbalanced data)

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Calculate scale_pos_weight for XGBoost
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

top3_models = {
    "LightGBM": Pipeline([
        ("prep", preprocessor),
        ("model", LGBMClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            class_weight="balanced", random_state=RANDOM_SEED,
            verbose=-1, n_jobs=-1
        ))
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("model", XGBClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            scale_pos_weight=scale_pos_weight, random_state=RANDOM_SEED,
            eval_metric="logloss", verbosity=0, n_jobs=-1
        ))
    ]),
    "GradientBoosting": Pipeline([
        ("prep", preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=5,
            random_state=RANDOM_SEED
        ))
    ]),
}

top3_results = []
top3_fitted = {}
for name, pipe in top3_models.items():
    r = evaluate_model(name, pipe, X_train, X_test, y_train, y_test)
    top3_results.append(r)
    top3_fitted[name] = pipe

top3_df = pd.DataFrame(top3_results).sort_values("F1", ascending=False)
print("\nTop 3 Model Results:")
display(top3_df)

In [ ]:
# Confusion matrices for all top 3
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pipe) in zip(axes, top3_fitted.items()):
    y_pred = pipe.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=["No", "Yes"],
        cmap="Blues", ax=ax
    )
    ax.set_title(f"{name}")

plt.suptitle("Confusion Matrices — Top 3 Models", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves for all top 3
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, pipe in top3_fitted.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=axes[0], name=name)
axes[0].set_title("ROC Curves — Top 3 Models")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)

for name, pipe in top3_fitted.items():
    PrecisionRecallDisplay.from_estimator(pipe, X_test, y_test, ax=axes[1], name=name)
axes[1].set_title("Precision-Recall Curves — Top 3 Models")

plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification report for the best model
best_model_name = top3_df.iloc[0]["Model"]
best_pipe = top3_fitted[best_model_name]
y_pred_best = best_pipe.predict(X_test)

print(f"Best model: {best_model_name}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=["No Attrition", "Attrition"]))

## 23. Error Analysis

Understanding where the model fails is as important as knowing where it succeeds. We examine:
- False positives (predicted attrition but employee stayed)
- False negatives (predicted no attrition but employee left) — the more costly error

In [ ]:
# Error breakdown
y_pred_best = best_pipe.predict(X_test)
error_df = X_test.copy()
error_df["actual"] = y_test.values
error_df["predicted"] = y_pred_best
error_df["error_type"] = "Correct"
error_df.loc[(error_df["actual"] == 1) & (error_df["predicted"] == 0), "error_type"] = "False Negative"
error_df.loc[(error_df["actual"] == 0) & (error_df["predicted"] == 1), "error_type"] = "False Positive"

print("Error breakdown:")
print(error_df["error_type"].value_counts())

# Compare feature means for FN vs correct positives
fn = error_df[error_df["error_type"] == "False Negative"]
tp = error_df[(error_df["actual"] == 1) & (error_df["predicted"] == 1)]

if len(fn) > 0 and len(tp) > 0:
    compare_cols = ["MonthlyIncome", "YearsAtCompany", "DistanceFromHome",
                    "JobSatisfaction", "OverTime", "TotalWorkingYears"]
    compare_cols = [c for c in compare_cols if c in error_df.columns and
                    error_df[c].dtype in [np.float64, np.int64, float, int]]
    if compare_cols:
        comparison = pd.DataFrame({
            "False Negative (mean)": fn[compare_cols].mean(),
            "True Positive (mean)": tp[compare_cols].mean(),
        })
        print("\nFN vs TP feature comparison:")
        display(comparison)

## 24. Interpretation and Business Insight

We extract feature importances from the best tree-based model to understand which factors drive attrition predictions.

In [ ]:
# Feature importance from the best model
if hasattr(best_pipe.named_steps["model"], "feature_importances_"):
    importances = best_pipe.named_steps["model"].feature_importances_
    feature_names_out = best_pipe.named_steps["prep"].get_feature_names_out()

    imp_df = pd.DataFrame({
        "feature": feature_names_out,
        "importance": importances
    }).sort_values("importance", ascending=False)

    plt.figure(figsize=(12, 8))
    top_n = min(20, len(imp_df))
    sns.barplot(data=imp_df.head(top_n), x="importance", y="feature", palette="viridis")
    plt.title(f"Top {top_n} Feature Importances — {best_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()

    print("\nTop 10 features:")
    display(imp_df.head(10))
else:
    print("Model does not expose feature_importances_.")

### Business Insights

1. **OverTime** is typically the strongest predictor — employees working overtime are significantly more likely to leave.
2. **MonthlyIncome** and **JobLevel** — lower-paid, lower-level employees have higher attrition.
3. **YearsAtCompany / YearsWithCurrManager** — very short tenure correlates with higher attrition (new hires are at risk).
4. **StockOptionLevel** — employees with stock options are more likely to stay.
5. **DistanceFromHome** — longer commutes increase attrition risk.

**Actionable recommendations**:
- Review overtime policies, especially for at-risk roles
- Strengthen onboarding and early-career engagement
- Consider flexible work arrangements for long-commute employees
- Ensure competitive compensation at lower job levels

## 25. Limitations

1. **Synthetic data**: IBM created this dataset for demonstration — real HR data has far more noise and biases.
2. **Small sample**: 1,470 rows is tiny by production standards. Models may not generalize.
3. **No temporal dimension**: We cannot model tenure-based hazard rates (survival analysis would be better).
4. **Binary target**: Real attrition has shades — involuntary vs voluntary, retirement vs job-hop.
5. **No fairness audit**: Protected attributes (Age, Gender, MaritalStatus) are features — a production model must audit for bias.
6. **Single split evaluation**: A more rigorous evaluation would use k-fold cross-validation.

## 26. How to Improve This Project

1. **Cross-validation**: Use stratified k-fold instead of a single train/test split.
2. **Threshold tuning**: Optimize the decision threshold for business-specific cost functions.
3. **SHAP analysis**: Use SHAP values for more nuanced feature-level explanations.
4. **Survival analysis**: Model time-to-event (attrition) rather than binary classification.
5. **Fairness audit**: Check model predictions for bias across gender, age groups.
6. **Feature selection**: Use recursive feature elimination or Boruta to reduce dimensionality.
7. **Ensemble stacking**: Combine top 3 models with a meta-learner.
8. **External validation**: Test on a different company's HR data to assess generalizability.

## 27. Production Considerations

| Aspect | Recommendation |
|---|---|
| **Model format** | Export with joblib or ONNX |
| **Inference latency** | Tree models are fast (<1ms per prediction) |
| **Retraining cadence** | Monthly or quarterly as workforce changes |
| **Monitoring** | Track prediction distribution shift and actual attrition rates |
| **Fairness** | Audit predictions across protected groups before deployment |
| **Data privacy** | HR data is highly sensitive — encrypt, access-control, anonymize |
| **Human-in-the-loop** | Predictions should inform HR conversations, not automate decisions |

## 28. Common Mistakes

1. **Using accuracy as the primary metric** on imbalanced data — a dummy model gets ~84%.
2. **Fitting preprocessors on the full dataset** before splitting — causes data leakage.
3. **Ignoring ID columns** (EmployeeNumber) — the model memorizes IDs instead of learning patterns.
4. **Not handling zero-variance columns** — EmployeeCount, StandardHours, Over18 add noise.
5. **Treating ordinal features as continuous** without checking if the scale is meaningful.
6. **Using SMOTE on small datasets** — can introduce synthetic noise worse than class weighting.
7. **Selecting models based on accuracy alone** when the business cost of FN >> FP.
8. **Not comparing against a dummy baseline** — you can't claim a model is good without one.

## 29. Mini Challenge / Exercises

1. **Threshold tuning**: Find the optimal classification threshold that maximizes F2-score (heavier weight on recall). Plot the threshold vs. F2 curve.
2. **Feature ablation**: Remove the top 3 features and retrain — how much does performance drop?
3. **SHAP values**: Install `shap` and create a SHAP summary plot for the best model.
4. **Age bias audit**: Split test predictions by age group (< 30, 30-45, > 45) and compare recall across groups.
5. **Cost-sensitive evaluation**: Assume replacing an employee costs $50K and flagging incorrectly costs $1K. Calculate the expected cost of the best model vs. the dummy baseline.

## 30. Final Summary / Key Takeaways

1. **Attrition prediction is a classic imbalanced binary classification problem** — accuracy alone is misleading.
2. **Preprocessing matters**: Dropping ID/zero-variance columns, encoding categoricals, and scaling numerics are essential.
3. **Splitting before fitting** prevents data leakage — always do this.
4. **LazyPredict** quickly identifies promising model families (tree-based ensembles dominate tabular data).
5. **FLAML** finds well-tuned models automatically and often matches or beats manual tuning.
6. **Top 3 evaluation** with confusion matrices, ROC curves, and PR curves gives a complete picture.
7. **OverTime, MonthlyIncome, and tenure features** are the strongest attrition drivers.
8. **Business action**: Use model outputs to prioritize retention interventions, not to automate decisions.
9. **Always compare against a dummy baseline** — it's the most important sanity check in ML.